In [2]:
import os
import pandas as pd

data_dir = ["data/processed/"]


for dir in data_dir:
    print(f"Directory: {dir}")
    for file in sorted(os.listdir(dir)):
        if file.endswith(".csv"):
            file_path = os.path.join(dir, file)
            df = pd.read_csv(file_path)
            print(f"File: {file}")
            print(df.columns)
            print(df["Writer"].value_counts())

Directory: data/processed/
File: processed_articles.csv
Index(['Writer', 'Text', 'ppl', 'burstiness', 'ttr', 'complex', 'avg_word_len',
       'avg_sent_len', 'uid', 'gain',
       ...
       'bert_758', 'bert_759', 'bert_760', 'bert_761', 'bert_762', 'bert_763',
       'bert_764', 'bert_765', 'bert_766', 'bert_767'],
      dtype='object', length=791)
Writer
GPT-4o        7321
Yi-Large      7319
Mistral-7B    7316
Qwen-2-72B    7314
Gemma-2-9B    7310
Llama-8B      7306
Human         7295
Name: count, dtype: int64
File: processed_cross_domain_essay.csv
Index(['Text', 'Writer', 'is_AI', 'ppl', 'burstiness', 'syntax_depth', 'ttr',
       'semantic_mean', 'semantic_std', 'uid',
       ...
       'bert_761', 'bert_762', 'bert_763', 'bert_764', 'bert_765', 'bert_766',
       'bert_76***7', 'complex', 'avg_word_len', 'avg_sent_len'],
      dtype='object', length=791)
Writer
Human      200
GPT4All    196
Name: count, dtype: int64
File: processed_cross_domain_wp.csv
Index(['Text', 'Writer', 'i

In [ ]:
import pandas as pd
import re
from nltk.tokenize import sent_tokenize
import os

files_to_fix = [
    "data/processed/processed_cross_domain_essay.csv",
    "data/processed/processed_cross_domain_wp.csv",
    "data/processed/processed_translation.csv",
    "data/processed/processed_unseen_reuters.csv"
]

print("Fixing missing Stylometric Features...")

for file_path in files_to_fix:
    if not os.path.exists(file_path):
        continue
        
    df = pd.read_csv(file_path)
    
    # التأكد من أن الميزات غير موجودة قبل إضافتها
    if 'complex' not in df.columns:
        complex_ratios = []
        avg_word_lens = []
        avg_sent_lens = []
        
        # استخراج الميزات الأسلوبية
        for text in df['Text']:
            text_str = str(text)
            words = re.findall(r"\w+", text_str.lower())
            sentences = sent_tokenize(text_str)
            
            if words and sentences:
                avg_word_lens.append(sum(len(w) for w in words) / len(words))
                avg_sent_lens.append(len(words) / len(sentences))
                complex_ratios.append(len([w for w in words if len(w) > 6]) / len(words))
            else:
                avg_word_lens.append(0.0)
                avg_sent_lens.append(0.0)
                complex_ratios.append(0.0)
                
        df['complex'] = complex_ratios
        df['avg_word_len'] = avg_word_lens
        df['avg_sent_len'] = avg_sent_lens
        
        df.to_csv(file_path, index=False)
        print(f"Added 3 missing features to: {os.path.basename(file_path)}")
    else:
        print(f"Features already exist in: {os.path.basename(file_path)}")

    
print("All datasets are now 100% aligned and ready for training.")

Fixing missing Stylometric Features...
Features already exist in: processed_cross_domain_essay.csv
Features already exist in: processed_cross_domain_wp.csv
Features already exist in: processed_translation.csv
Features already exist in: processed_unseen_reuters.csv
Renamed main dataset to match train.ipynb
All datasets are now 100% aligned and ready for training.


In [ ]:
df.shape

In [ ]:
df.rename(columns={"Paraphrased_Text": "Article"}, inplace=True)

In [ ]:
import os
import pandas as pd

# المجلدات المستهدفة
directories = ["data/processed/", "data/need_processing/"]

# قاموس التعيين الشامل
CLEAN_MAPPING = {
    # الفئة البشرية
    'Human_story': 'Human',
    'Human_story_paraphrased': 'Human',
    'human': 'Human',

    # GPT-4o
    'GPT_4-o': 'GPT-4o',
    'GPT_4-o_paraphrased': 'GPT-4o',

    # Gemma
    'gemma-2-9b': 'Gemma-2-9B',
    'gemma-2-9b_paraphrased': 'Gemma-2-9B',

    # Mistral
    'mistral-7B': 'Mistral-7B',
    'mistral-7B_paraphrased': 'Mistral-7B',

    # Llama
    'llama-8B': 'Llama-8B',
    'llama-8B_paraphrased': 'Llama-8B',

    # Qwen
    'qwen-2-72B': 'Qwen-2-72B',
    'qwen-2-72B_paraphrased': 'Qwen-2-72B',

    # Yi-Large
    'accounts/yi-01-ai/models/yi-large': 'Yi-Large',
    'accounts/yi-01-ai/models/yi-large_paraphrased': 'Yi-Large'
}

print("=" * 80)
print("STANDARDIZING ALL WRITER NAMES AND COLUMNS")
print("=" * 80)

for directory in directories:
    if not os.path.exists(directory):
        continue
        
    for file in os.listdir(directory):
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(directory, file)
        df = pd.read_csv(file_path)

        # 1. توحيد اسم العمود ليكون 'Writer' دائماً
        if 'Writers' in df.columns:
            df = df.rename(columns={'Writers': 'Writer'})

        # 2. تطبيق قاموس التعيين لتنظيف أسماء الكتاب
        if 'Writer' in df.columns:
            df['Writer'] = df['Writer'].map(CLEAN_MAPPING).fillna(df['Writer'])

        # حفظ الملف
        df.to_csv(file_path, index=False)
        print(f"✓ Processed & Cleaned: {directory}{file}")

print("\n" + "=" * 80)
print("VERIFICATION COMPLETE - ALL FILES STANDARDIZED")
print("=" * 80)